# Einlesen der Buspassagierdaten

## 1. Bibliotheken importieren

In [1]:
## 1) Import der notwendigen Bibliotheken
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
import traceback
import datediff
import os

## 2. Spark Session erstellen

In [2]:
# Falls noch keine Session existiert, wird sie erstellt.
spark = SparkSession.builder.appName("CSVToParquetProcessing").getOrCreate()

## 3. Pfade und Schemas definieren

In [ ]:

def list_csv_files(dir_path: str) -> list:
    """
    Listet alle .csv-Dateien im angegebenen lokalen Verzeichnis (nicht rekursiv)
    und gibt eine Liste der vollständigen Pfade zurück.
    """
    return [
        os.path.join(dir_path, f)
        for f in os.listdir(dir_path)
        if f.lower().endswith(".csv") and os.path.isfile(os.path.join(dir_path, f))
    ]

directory = "data/raw_data"  # Passe diesen Pfad an
file_paths = list_csv_files(directory)

print(file_paths)


In [4]:
column_names = [
    "period", "weekday", "date", "device", "line", "direction", "day_type",
    "station_number", "stop_name", "boarders", "alighters", "occupancy",
    "capacity", "utilization", "planned_arrival_time", "planned_departure_time",
    "actual_arrival", "actual_departure", "stop_time", "delay"
]

## 4. Funktion(en) zum Einlesen einzelner CSV-Dateien

In [5]:
def process_csv_file(file_path: str, delimiter: str = ';', encoding: str = 'utf-8') -> DataFrame:
    """
    Reads a CSV file, processes it by removing unnecessary columns, and renames columns based on the expected format.
    
    Args:
    - file_path: Path to the CSV file
    - delimiter: Character separating fields in the file
    - encoding: File encoding type
    
    Returns:
    - DataFrame: Processed PySpark DataFrame
    """
    try:
        if not file_path:
            raise ValueError(f"Invalid file path: {file_path}")
        
        # CSV-Datei einlesen
        data = spark.read.option("encoding", encoding).csv(file_path, sep=delimiter, header=False, inferSchema=True)
        
        # Debugging: Anzahl der ursprünglichen Spalten ausgeben
        num_columns = len(data.columns)
        print(f"Original number of columns: {num_columns}")
        
        # Entfernen der ersten und letzten Spalte, falls mehr als zwei Spalten vorhanden sind
        if num_columns > 2:
            data = data.select(*data.columns[1:num_columns - 1])
        
        # Umbenennen der Spalten, falls die Anzahl übereinstimmt
        if len(column_names) == len(data.columns):
            for old_col, new_col in zip(data.columns, column_names):
                data = data.withColumnRenamed(old_col, new_col)
            print(f"Renamed columns to: {column_names}")
        else:
            print(f"Warning: Number of columns ({len(data.columns)}) does not match expected number ({len(column_names)}).")
        
        return data
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        traceback.print_exc()
        return None

## 5. Zusammenführen der CSV-Dateien

In [6]:
# Leerer DataFrame mit vorgegebenem Schema
final_df = spark.createDataFrame([], schema=spark.createDataFrame([tuple([''] * len(column_names))], column_names).schema)

# Iteration über alle CSV-Dateien und Unionierung der Ergebnisse
for file in file_paths:
    new_df = process_csv_file(file)
    if new_df is not None:
        final_df = final_df.union(new_df)
    else:
        print(f"Skipping file {file} due to processing errors.")

# Debugging: Gesamtanzahl der Zeilen ausgeben
print(f"Total number of rows in final DataFrame: {final_df.count()}")
final_df.show(5)

In [9]:
# 1) Anzahl der Zeilen
row_count = final_df.count()
print(f"Anzahl Zeilen: {row_count}")

# 2) Minimal- und Maximal-Datum
stats = (
    final_df
    .select(
        min("date").alias("min_date"),
        max("date").alias("max_date")
    )
    .collect()[0]
)
min_date = stats["min_date"]
max_date = stats["max_date"]
print(f"Zeitspanne von {min_date} bis {max_date}")

# 3) Dauer in Tagen zwischen erstem und letztem Datum
span_days = (
    final_df
    .select(
        datediff(max("date"), min("date")).alias("span_days")
    )
    .collect()[0]["span_days"]
)
print(f"Dauer: {span_days} Tage")


## 6. Speichern des kombinierten DataFrames als Parquet

In [7]:
# Zielpfad für die Speicherung
save_path = "data/imported_data"

# Speichern als Parquet-Datei mit Überschreiben existierender Daten
final_df.write.mode("overwrite").parquet(save_path)
print(f"Data successfully saved to: {save_path}")